# Post-Processing Threshold Tuning

The ensemble bottom detector feeds its output through `post_process_bottom()`, which flags pings that deviate more than `max_deviation_m` meters from a sliding median over `window` pings. The current defaults (10 m over 31 pings) are conservative — on sloping or rough bathymetry, real bottom changes can easily exceed that. This notebook sweeps both parameters and visualises exactly which pings get flagged, so we can pick values that only catch genuine spikes.

In [ ]:
import sys
sys.path.insert(0, '../..')
from project_paths import REPO_ROOT

import numpy as np
import matplotlib.pyplot as plt

from raw_sensor_integration.data_loader import load_raw
from raw_sensor_integration.bottom_detection import (
    detect_bottom_ensemble_pipeline,
)
from raw_sensor_integration.bottom_labelling.ensemble import post_process_bottom

## Load echograms

In [ ]:
RAW_DIR = REPO_ROOT / 'Datasets/raw_files/drive-download-20260317T113707Z-1-001'
all_raw = sorted(RAW_DIR.glob('*.raw'))
print(f'Found {len(all_raw)} raw files')

rng = np.random.default_rng(seed=42)
n_files = min(5, len(all_raw))
selected_idx = rng.choice(len(all_raw), size=n_files, replace=False)
selected_files = [all_raw[i] for i in sorted(selected_idx)]

for f in selected_files:
    print(f'  {f.name}')

In [ ]:
datasets = []

for raw_path in selected_files:
    print(f'Loading {raw_path.name}...')
    data = load_raw(raw_path, frequency=38000)

    # Run ensemble without post-processing to get the raw trace
    ensemble_raw, _, details = detect_bottom_ensemble_pipeline(
        data.sv_db, data.range_array,
        heave=data.heave,
        pulse_duration=data.pulse_duration,
        frequency_hz=data.frequency_hz,
        post_process=False,
    )

    datasets.append({
        'name': raw_path.stem,
        'power_db': data.sv_db,
        'range_arr': data.range_array,
        'ensemble_raw': ensemble_raw,
    })
    print(f'  {data.sv_db.shape[1]} pings, {data.range_array[-1]:.0f}m max range')

print(f'\nLoaded {len(datasets)} echograms.')

## Threshold sweep

Each row shows the same echogram post-processed at a different `max_deviation_m`. The red line is the cleaned bottom trace; orange dots mark pings that were flagged as outliers and replaced by interpolation.

In [ ]:
THRESHOLDS = [5, 10, 15, 20, 30, 50]
WINDOW = 31  # fixed window for this sweep

for ds in datasets:
    n_pings = ds['power_db'].shape[1]
    ping_idx = np.arange(n_pings)
    raw = ds['ensemble_raw']

    fig, axes = plt.subplots(len(THRESHOLDS), 1, figsize=(16, 4 * len(THRESHOLDS)),
                             sharex=True, sharey=True)

    for ax, thresh in zip(axes, THRESHOLDS):
        cleaned, outlier_mask = post_process_bottom(
            raw, max_deviation_m=thresh, window=WINDOW, fill_gaps=True,
        )
        n_outliers = outlier_mask.sum()

        ax.imshow(
            ds['power_db'], aspect='auto', origin='upper',
            cmap='viridis', vmin=-80, vmax=-20,
            extent=[0, n_pings, ds['range_arr'][-1], ds['range_arr'][0]],
        )
        ax.plot(ping_idx, cleaned, color='red', linewidth=1.5, label='Cleaned bottom')

        outlier_pings = np.where(outlier_mask)[0]
        if len(outlier_pings) > 0:
            ax.scatter(outlier_pings, raw[outlier_pings], color='orange', s=20,
                       zorder=5, label=f'Outliers ({n_outliers})')

        ax.set_ylabel('Range (m)')
        ax.set_title(f'max_deviation = {thresh} m — {n_outliers} outliers flagged',
                     fontsize=10)
        ax.legend(loc='upper right', fontsize=8, framealpha=0.8)

    axes[-1].set_xlabel('Ping number')
    fig.suptitle(ds['name'], fontsize=13, y=1.0)
    plt.tight_layout()
    plt.show()

## Window size sweep

Same idea, but now `max_deviation_m` is fixed at 10 m (the current default) and we vary the window width. A wider window means the local median is computed over more pings, which smooths out the reference but also makes it slower to follow real slopes.

In [ ]:
WINDOWS = [11, 21, 31, 51, 71]
THRESHOLD = 10.0  # fixed threshold for this sweep

for ds in datasets:
    n_pings = ds['power_db'].shape[1]
    ping_idx = np.arange(n_pings)
    raw = ds['ensemble_raw']

    fig, axes = plt.subplots(len(WINDOWS), 1, figsize=(16, 4 * len(WINDOWS)),
                             sharex=True, sharey=True)

    for ax, win in zip(axes, WINDOWS):
        cleaned, outlier_mask = post_process_bottom(
            raw, max_deviation_m=THRESHOLD, window=win, fill_gaps=True,
        )
        n_outliers = outlier_mask.sum()

        ax.imshow(
            ds['power_db'], aspect='auto', origin='upper',
            cmap='viridis', vmin=-80, vmax=-20,
            extent=[0, n_pings, ds['range_arr'][-1], ds['range_arr'][0]],
        )
        ax.plot(ping_idx, cleaned, color='red', linewidth=1.5, label='Cleaned bottom')

        outlier_pings = np.where(outlier_mask)[0]
        if len(outlier_pings) > 0:
            ax.scatter(outlier_pings, raw[outlier_pings], color='orange', s=20,
                       zorder=5, label=f'Outliers ({n_outliers})')

        ax.set_ylabel('Range (m)')
        ax.set_title(f'window = {win} pings — {n_outliers} outliers flagged',
                     fontsize=10)
        ax.legend(loc='upper right', fontsize=8, framealpha=0.8)

    axes[-1].set_xlabel('Ping number')
    fig.suptitle(ds['name'], fontsize=13, y=1.0)
    plt.tight_layout()
    plt.show()

## Outlier counts across all combinations

In [ ]:
thresholds = [5, 10, 15, 20, 30, 50]
windows = [11, 21, 31, 51, 71]

# Build outlier count matrix per dataset
for ds in datasets:
    raw = ds['ensemble_raw']
    n_pings = len(raw)
    counts = np.zeros((len(thresholds), len(windows)), dtype=int)

    for i, thresh in enumerate(thresholds):
        for j, win in enumerate(windows):
            _, mask = post_process_bottom(raw, max_deviation_m=thresh, window=win, fill_gaps=False)
            counts[i, j] = mask.sum()

    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(counts, aspect='auto', cmap='YlOrRd')
    ax.set_xticks(range(len(windows)))
    ax.set_xticklabels(windows)
    ax.set_yticks(range(len(thresholds)))
    ax.set_yticklabels(thresholds)
    ax.set_xlabel('Window size (pings)')
    ax.set_ylabel('max_deviation (m)')
    ax.set_title(f'{ds["name"]} — outlier count (of {n_pings} pings)')

    for i in range(len(thresholds)):
        for j in range(len(windows)):
            ax.text(j, i, str(counts[i, j]), ha='center', va='center',
                    color='white' if counts[i, j] > counts.max() / 2 else 'black',
                    fontsize=9)

    plt.colorbar(im, ax=ax, label='Outliers flagged')
    plt.tight_layout()
    plt.show()